## Homework 2

In [1]:
import os
from dotenv import load_dotenv

load_dotenv ()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:4])

sk-p


### Embedding the query

In [2]:
from fastembed import TextEmbedding

In [3]:
model = "jinaai/jina-embeddings-v2-small-en"
query = 'I just discovered the course. Can I join now?'

In [39]:
embedding_model = (TextEmbedding(model))

In [20]:
embeddings_generator = embedding_model.embed(query)

In [21]:
embeddings_list = list(embeddings_generator)

In [22]:
len(embeddings_list[0])

512

In [24]:
min(embeddings_list[0])

-0.11726373885183883

#### What's the minimal value in this array?
 
* -0.11

### Q2. Cosine similarity with another vector

In [35]:
doc = 'Can I still join the course after the start date?'

In [40]:
doc_embedding = embedding_model.embed(doc)

In [45]:
doc_embedding_list = list(doc_embedding)

In [46]:
len(doc_embedding_list[0])

512

In [48]:
import numpy as np 

In [50]:
np.array(doc_embedding_list[0]).dot(np.array(embeddings_list[0]))

0.9008528895674548

What's the cosine similarity between the vector for the query
and the vector for the document?
 
* 0.9

###  Q3. Ranking by cosine

In [2]:
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]

In [1]:
from qdrant_client import QdrantClient, models

qd_client  = QdrantClient("http://localhost:6333")


In [3]:
VECTOR_SIZE = 512 
COLLECTION_NAME = "my_documents"
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [4]:
qd_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,                 
        distance=models.Distance.COSINE),
)

True

In [5]:
points = [] 

for i, doc in enumerate(documents): 
    text =   doc['text']
    vector = models.Document(text=text, model=model_handle) #embed text locally with "jinaai/jina-embeddings-v2-small-en" from FastEmbed 
    point = models.PointStruct(
        id=i,
        vector= vector,
        payload= doc
    )
    points.append(point)
 

In [8]:
qd_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [9]:
from fastembed import TextEmbedding

In [ ]:
import numpy as np

embedding_model = TextEmbedding(model_handle)
query = 'I just discovered the course. Can I join now?'

query_vector = np.array(list(embedding_model.embed(query))[0]) 

In [21]:
len(query_vector)

512

In [23]:
question = 'I just discovered the course. Can I join now?'


In [24]:
responses = qd_client.query_points(
            collection_name=COLLECTION_NAME,
            query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
                text=question,
                model=model_handle 
            ),
            limit=1, # top closest matches
            with_payload=True #to get metadata in the results
        )

In [31]:
responses.points[0].id

1

In [32]:
print(responses.points[0].id)

1


What's the document index with the highest similarity? (Indexing starts from 0):
 
- 1 

### Q4. Ranking by cosine, version twov

In [37]:
points = [] 

for i, doc in enumerate(documents): 
    text =   doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle) #embed text locally with "jinaai/jina-embeddings-v2-small-en" from FastEmbed 
    point = models.PointStruct(
        id=i,
        vector= vector,
        payload= doc
    )
    points.append(point)
 

In [38]:
qd_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [39]:
responses = qd_client.query_points(
            collection_name=COLLECTION_NAME,
            query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
                text=question,
                model=model_handle 
            ),
            limit=1, # top closest matches
            with_payload=True #to get metadata in the results
        )

In [40]:
print(responses.points[0].id)

0


Embed this field and compute the cosine between it and the
query vector. What's the highest scoring document?

- 0 

### Q5. Selecting the embedding model

In [78]:
models = TextEmbedding.list_supported_models()

In [70]:
(model)

[{'model': 'BAAI/bge-base-en',
  'sources': {'hf': 'Qdrant/fast-bge-base-en',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.42,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model': 'BAAI/bge-base-en-v1.5',
  'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.21,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model':

In [ ]:
dim = 768
for model in models:
    if model['dim'] < dim:
        dim = model['dim'] 

384


What's the smallest dimensionality for models in fastembed?
 
- 384 


### Q6. Indexing with qdrant (2 points)


In [82]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [85]:
from qdrant_client import QdrantClient, models

In [88]:
model_handle = 'BAAI/bge-small-en'

In [86]:
qdrant_client = QdrantClient("http://localhost:6333")

In [87]:
# Define the collection name
collection_name = "mlzoomcamp-rag"
EMBEDDING_DIMENSIONALITY = 384

# Create the collection with specified vector parameters
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,  # Dimensionality of the vectors
        distance=models.Distance.COSINE  # Distance metric for similarity search
    )
)

True

In [90]:
points = [] 

for i, doc in enumerate(documents): 
    text = doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle) #embed text locally with "jinaai/jina-embeddings-v2-small-en" from FastEmbed 
    point = models.PointStruct(
        id=i,
        vector= vector,
        payload= doc
    )
    points.append(point)
 

In [91]:
len(points)

375

In [92]:
qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model_optimized.onnx:   0%|          | 0.00/133M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

c:\Users\crab\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\crab\AppData\Local\Temp\fastembed_cache\models--Qdrant--bge-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [94]:
question = 'I just discovered the course. Can I join now?'

In [95]:
results = qdrant_client.query_points(
        collection_name=collection_name,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=question,
            model=model_handle 
        ),
        limit=1, # top closest matches
        with_payload=True #to get metadata in the results
    )

In [96]:
print(results)

points=[ScoredPoint(id=14, version=0, score=0.8703172, payload={'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'section': 'General course-related questions', 'question': 'The course has already started. Can I still join it?', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None)]


What's the highest score in the results?
(The score for the first returned record):
 
- 0.87 